# Diabetes Prediction — Machine Learning Project


### Project objective
Build a classification model that predicts whether a person is likely to have diabetes (`Outcome = 1`) or not (`Outcome = 0`) using the Pima Indians Diabetes dataset.

### Complete workflow
1. Import libraries
2. Load dataset
3. Understand the dataset
4. Check data quality
5. Exploratory Data Analysis (EDA)
6. Handle invalid/missing values
7. Separate features and target
8. Train-test split
9. Feature scaling
10. Train multiple classification models
11. Evaluate models
12. Compare models
13. Select the best model
14. Test with a new patient
15. Save the model and scaler
16. Verify the saved model



## Step 1 — Import libraries

We will use:
- **Pandas** for data handling
- **NumPy** for numerical operations
- **Matplotlib / Seaborn** for visualization
- **Scikit-learn** for machine learning
- **Joblib** to save the trained model and scaler


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


Libraries imported successfully!


## Step 2 — Load the dataset


In [ ]:
# Load dataset
df = pd.read_csv('/Users/juweria/Desktop/untitled folder/diabetes. csv.csv')

print("Dataset loaded successfully!")
print("Shape:", df.shape)


## Step 3 — View the first records

`head()` helps us understand the structure and values in the dataset.


In [ ]:
df.head()


## Step 4 — Understand rows and columns

- `shape` → number of rows and columns
- `columns` → column names
- `dtypes` → data types


In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)


## Step 5 — Check dataset information

`info()` tells us about:
- number of records
- data types
- non-null values


In [ ]:
df.info()


## Step 6 — Statistical summary

`describe()` gives statistics such as mean, standard deviation, minimum, maximum and quartiles.


In [ ]:
df.describe().T


## Step 7 — Check missing values

Before training a model, always check whether data is missing.


In [ ]:
missing = df.isnull().sum()
print(missing)


## Step 8 — Check duplicate rows


In [ ]:
print("Duplicate rows:", df.duplicated().sum())


## Step 9 — Check target distribution

`Outcome` is our target:
- `0` → No diabetes
- `1` → Diabetes

We should understand whether the classes are balanced.


In [ ]:
print(df["Outcome"].value_counts())

sns.countplot(x="Outcome", data=df)
plt.title("Diabetes Outcome Distribution")
plt.xlabel("Outcome (0 = No Diabetes, 1 = Diabetes)")
plt.ylabel("Count")
plt.show()


## Step 10 — Important data-quality check

In this dataset, some medical measurements contain `0`, which is not physiologically meaningful for variables such as glucose, blood pressure and BMI.

We will treat these zeros as missing values for:
- Glucose
- BloodPressure
- SkinThickness
- Insulin
- BMI

We will **not** replace zeros in `Pregnancies` or `Outcome`, because zero is meaningful there.


In [ ]:
invalid_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

for col in invalid_zero_columns:
    print(f"{col}: {(df[col] == 0).sum()} zero values")


## Step 11 — Replace invalid zeros with NaN


In [ ]:
df[invalid_zero_columns] = df[invalid_zero_columns].replace(0, np.nan)

print("Missing values after replacing invalid zeros:")
print(df.isnull().sum())


## Step 12 — Fill missing values with the median

For this beginner project, we will use the median of each feature.

Why median?
- It is less affected by extreme values than the mean.
- It is simple to explain to beginners.


In [ ]:
for col in invalid_zero_columns:
    df[col] = df[col].fillna(df[col].median())

print("Missing values after imputation:")
print(df.isnull().sum())


## Step 13 — Exploratory Data Analysis: distributions

Look at the distribution of the main numerical features.


In [ ]:
df.hist(figsize=(12, 10), bins=20)
plt.tight_layout()
plt.show()


## Step 14 — Correlation analysis

Correlation helps us understand how numerical variables move in relation to each other.

> Correlation does not mean causation.


In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


## Step 15 — Compare glucose levels by outcome

This is a useful EDA question:
**Do the glucose distributions look different for the two outcome groups?**


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="Outcome", y="Glucose", data=df)
plt.title("Glucose vs Diabetes Outcome")
plt.xlabel("Outcome")
plt.ylabel("Glucose")
plt.show()


## Step 16 — Compare BMI by outcome


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="Outcome", y="BMI", data=df)
plt.title("BMI vs Diabetes Outcome")
plt.xlabel("Outcome")
plt.ylabel("BMI")
plt.show()


## Step 17 — Define features (X) and target (y)

**X** = input variables used by the model.

**y** = output we want to predict.


In [ ]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## Step 18 — Train-test split

We will use:
- **80%** for training
- **20%** for testing

`stratify=y` keeps the class proportion similar in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


## Step 19 — Feature scaling

Algorithms such as Logistic Regression and KNN can benefit from scaling.

We use `StandardScaler`.

**Very important:**
- `fit_transform()` → training data
- `transform()` → test data

This prevents information from the test set leaking into training.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")


# Step 20 — Model 1: Logistic Regression

Logistic Regression is a strong baseline for binary classification.

It predicts the probability of belonging to class 0 or class 1.


In [ ]:
log_model = LogisticRegression(random_state=42)

log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, log_pred))


## Step 21 — Logistic Regression evaluation


In [ ]:
print(classification_report(y_test, log_pred))


# Step 22 — Model 2: Decision Tree

Decision Tree makes decisions using a sequence of rules.

It is easy to explain visually and does not require feature scaling in principle. We are still keeping the same prepared workflow here for comparison.


In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)

print("Decision Tree")
print("Accuracy:", accuracy_score(y_test, tree_pred))


## Step 23 — Decision Tree evaluation


In [ ]:
print(classification_report(y_test, tree_pred))


# Step 24 — Model 3: Random Forest

Random Forest combines many decision trees and usually provides a stronger baseline than a single tree.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, rf_pred))


## Step 25 — Random Forest evaluation


In [ ]:
print(classification_report(y_test, rf_pred))


# Step 26 — Model 4: K-Nearest Neighbors (KNN)

KNN predicts a class using nearby training examples.

KNN is sensitive to feature scale, which is why we use the scaled data.


In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)

knn_pred = knn_model.predict(X_test_scaled)

print("KNN")
print("Accuracy:", accuracy_score(y_test, knn_pred))


## Step 27 — Compare all models

We will compare accuracy, precision, recall and F1-score.

For a healthcare-style classification problem, students should understand that **accuracy alone is not enough**. Recall and precision can also matter.


In [ ]:
models = {
    "Logistic Regression": log_pred,
    "Decision Tree": tree_pred,
    "Random Forest": rf_pred,
    "KNN": knn_pred
}

results = []

for name, predictions in models.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df.sort_values("F1 Score", ascending=False)


## Step 28 — Visualize model comparison


In [ ]:
results_plot = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score"]]

results_plot.plot(kind="bar", figsize=(11, 6))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


# Step 29 — Confusion Matrix for Random Forest

A confusion matrix shows:
- True Negative
- False Positive
- False Negative
- True Positive

This is important because different types of errors have different meanings.


In [ ]:
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Diabetes", "Diabetes"],
    yticklabels=["No Diabetes", "Diabetes"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()


## Step 30 — Feature importance

Random Forest can show which features contributed most to its decisions.

This is useful for explaining the model, but feature importance should not be interpreted as proof of medical causation.


In [ ]:
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance)


## Step 31 — Plot feature importance


In [ ]:
plt.figure(figsize=(9, 5))
feature_importance.sort_values().plot(kind="barh")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.show()


# Step 32 — Select the model

For demonstration, we will use **Random Forest** as the final model.

In a real project, do not automatically choose a model only because it has the highest accuracy. Consider:
- recall
- precision
- F1-score
- cross-validation
- business/clinical cost of errors
- interpretability

For this classroom project, Random Forest is a convenient final model to demonstrate.


In [ ]:
final_model = rf_model

print("Final model selected: Random Forest")


# Step 33 — Predict a new patient

The input order must match the training feature order:

1. Pregnancies
2. Glucose
3. BloodPressure
4. SkinThickness
5. Insulin
6. BMI
7. DiabetesPedigreeFunction
8. Age

Because the final Random Forest model was trained on **unscaled X_train**, we do not scale this input.


In [ ]:
new_patient = pd.DataFrame([{
    "Pregnancies": 2,
    "Glucose": 120,
    "BloodPressure": 70,
    "SkinThickness": 25,
    "Insulin": 100,
    "BMI": 30.5,
    "DiabetesPedigreeFunction": 0.4,
    "Age": 35
}])

new_prediction = final_model.predict(new_patient)[0]
new_probability = final_model.predict_proba(new_patient)[0][1]

print("Prediction:", new_prediction)
print("Probability of class 1:", round(new_probability, 3))

if new_prediction == 1:
    print("Model prediction: Diabetes")
else:
    print("Model prediction: No Diabetes")


# Step 34 — Save the trained model

We will save:
- the trained Random Forest model
- the scaler used for Logistic Regression/KNN demonstrations
- the feature column order

The scaler is not required for our selected Random Forest model, but saving preprocessing objects is a good professional practice and will be useful when students later build other model versions.


In [ ]:
joblib.dump(final_model, "diabetes_model.pkl")
joblib.dump(scaler, "diabetes_scaler.pkl")
joblib.dump(list(X.columns), "diabetes_features.pkl")

print("Model saved as: diabetes_model.pkl")
print("Scaler saved as: diabetes_scaler.pkl")
print("Feature list saved as: diabetes_features.pkl")


# Step 35 — Verify the saved model

A good practice is to load the saved model again and check that it can make a prediction.


In [ ]:
loaded_model = joblib.load("diabetes_model.pkl")
loaded_features = joblib.load("diabetes_features.pkl")

loaded_prediction = loaded_model.predict(new_patient[loaded_features])[0]

print("Loaded model prediction:", loaded_prediction)
